# OsteoLab — 3. Inferencia

Tercer y último cuaderno de la serie. Hasta ahora teníamos: datos (cuaderno 1) y modelos ya
entrenados y evaluados con honestidad (cuaderno 2). Aquí cerramos el círculo: **¿qué pasa
exactamente cuando alguien sube una foto nueva de un hueso y pide una respuesta?**

Vamos a ver cómo se combinan los modelos en una cascada de decisiones (algunas muy baratas, otras
más caras), y cómo todo esto se expone como una API real que una aplicación web puede llamar.

## 0. Arrancamos (igual que en los cuadernos anteriores)

In [ ]:
import os
import sys
from pathlib import Path


def find_repo_root():
    # Sube desde el directorio actual hasta encontrar params.yaml (raíz del repo).
    start = Path.cwd()
    for candidate in (start, *start.parents):
        if (candidate / "params.yaml").exists() and (candidate / "training").is_dir():
            return candidate
    raise RuntimeError("No encuentro la raíz del repo. Abre el notebook desde dentro de él.")


REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

BONE_DB = REPO_ROOT / "base_datos_osea"

print("Raíz del proyecto:", REPO_ROOT)

In [ ]:
import subprocess


def run(cmd, enabled, cwd=REPO_ROOT):
    # Igual que en los cuadernos anteriores: lanza el comando real, o solo lo imprime si enabled=False.
    printable = " ".join(str(c) for c in cmd)
    if not enabled:
        print(f"[celda desactivada, el comando sería]\n$ {printable}")
        return None
    print(f"$ {printable}\n")
    proc = subprocess.run([sys.executable, *cmd[1:]] if cmd[0] == "python" else cmd,
                          cwd=cwd, text=True, capture_output=True)
    print(proc.stdout[-8000:])
    if proc.returncode != 0:
        print("STDERR:\n", proc.stderr[-4000:])
    return proc


RUN_TRAIN_CNN = False   # para la celda de la sección 2 que entrena una rama de etapa 2
RUN_EVALUATE = False

In [ ]:
# Reutilizamos la misma imagen de ejemplo que en el cuaderno 1.
sample = next(iter(sorted((BONE_DB / "renders_lab").rglob("*.png"))), None)
if sample is None:
    sample = next(iter(sorted((BONE_DB / "Fotos entrenamiento" / "_curated").rglob("*.jpg"))))
print("Imagen de ejemplo:", sample.relative_to(REPO_ROOT))

## 1. Primer nivel de decisión: una regla barata antes de gastar la CNN

No todo necesita un modelo caro. Para el grupo `hueso_largo`, la feature `elongation` que vimos en
el cuaderno 1 separa tan limpiamente que **no hace falta ejecutar la CNN en absoluto**: si
`elongation > 0.900`, la respuesta ya es `hueso_largo`, calculado con geometría clásica en
milisegundos. Ese umbral viene de un "stump" (un árbol de decisión de una sola pregunta) ajustado
sobre datos de entrenamiento.

¿Por qué justo `elongation` y no otra medida? Se probaron varias puertas candidatas — `n_holes`
para distinguir sacro/vértebra, `circularity`/`solidity` para cráneo/plano — y todas parecían
igual de buenas en el banco **sintético** (entre 88% y 92% de acierto). La diferencia apareció al
probarlas en **fotos reales**: `elongation` se mantuvo en 93% de acierto con solo 1 falso positivo
en 47 casos, mientras que las otras se desplomaron hasta un 16% de acierto, con 46 falsos
positivos en 49 casos. Es el mismo patrón sim2real del cuaderno 1: una medida puede parecer
perfecta en datos sintéticos y fallar por completo en el mundo real. Por eso, cualquier regla
nueva que se quiera añadir aquí tiene que validarse igual: accuracy **y** falsos positivos, medidos
sobre el banco de fotos reales, nunca solo sobre el sintético.

In [ ]:
from shared.inference.predict_cascade import ELONGATION_THRESHOLD, classify_image as cascade_classify

print("Umbral de elongación:", ELONGATION_THRESHOLD)

with open(sample, "rb") as fh:
    resultado = cascade_classify(fh, model_type="cnn_only", backend="classic")

# "gate" dice qué camino se tomó: "elongation_rule" (la regla barata) o "cnn" (se usó el modelo).
print({k: v for k, v in resultado.items() if k != "cv_features"})

## 2. Segundo nivel: afinar hueso concreto o taxón

Una vez que sabemos el grupo morfológico (cráneo, hueso largo, vértebra...), a veces queremos
afinar más: ¿qué hueso largo exactamente? ¿de qué taxón es este cráneo?

Aquí hubo una decisión de diseño importante. La primera idea — un único modelo gigante que
clasificase las 21 clases finas de golpe — no funcionaba bien: ramas sin relación entre sí se
interferían durante el entrenamiento (la clase `hueso_plano`, en particular, colapsaba). La
solución que está en producción es una **cascada de dos etapas**: la etapa 1 decide el grupo (lo
que acabamos de ver), y según qué grupo salió, se invoca un modelo *especializado* solo en ese
grupo — mucho más fácil de entrenar bien porque solo tiene que distinguir unas pocas clases
relacionadas entre sí.

| Grupo de etapa 1 | Modelo de etapa 2 que se invoca | Qué afina |
|---|---|---|
| `hueso_largo` | `stage2_hueso_largo` | fémur / húmero / ... |
| `hueso_pequeno` | `stage2_hueso_pequeno` | hueso fino de mano o pie |
| `vertebra` | `stage2_vertebra` | cervical / torácica / lumbar |
| `cranio` | `stage2_taxon_cranio` | taxón: humano / carnívoro / ungulado / roedor / otro primate |
| `mandibula_maxilar` | `stage2_taxon_mandibula_maxilar` | taxón |

Los grupos que todavía no tienen modelo de etapa 2 (`costilla`, `hueso_plano`, `pelvis`, `sacro`)
devuelven explícitamente `stage2_pending: true`. Esto es deliberado: distingue con claridad "aquí
no aplica ninguna etapa 2" de "falta entrenar esto todavía" — un `None` silencioso confundiría las
dos situaciones.

In [ ]:
from shared.inference.predict_two_stage import (STAGE2_MODELS, STAGE2_TAXON_MODELS,
                                                classify_image_two_stage)

print("Modelos de hueso fino:", STAGE2_MODELS)
print("Modelos de taxón:     ", STAGE2_TAXON_MODELS)

with open(sample, "rb") as fh:
    resultado = classify_image_two_stage(fh, backend="classic")

print({k: v for k, v in resultado.items() if k != "cv_features"})

In [ ]:
# Entrenar una rama de etapa 2 es exactamente la misma receta de entrenamiento del cuaderno 2,
# solo que con un dataset y un target distintos (celda desactivada por defecto).
run(["python", "-m", "training.train_cnn",
     "--model-type", "hybrid",
     "--dataset-path", "data/processed/bone_geometric_features_stage2_vertebra.csv",
     "--target", "bone_fine",
     "--run-name", "stage2_vertebra"], RUN_TRAIN_CNN)

run(["python", "-m", "training.evaluate_cascade_real_photos",
     "--model-type", "cnn_only", "--segmenter", "grounded_sam"], RUN_EVALUATE)

## 3. Las tres rutas de inferencia, lado a lado

En producción conviven, de hecho, **tres caminos distintos** para clasificar una imagen, porque
sirven a necesidades distintas de la aplicación:

1. **Geométrico** (Random Forest + proyección PCA 2D) — rápido, y además da un punto en el mapa
   PCA que la interfaz usa para mostrar "dónde cae" esta foto respecto a las demás.
2. **Cascada** — la regla barata de elongación + la CNN cuando hace falta (sección 1).
3. **Dos etapas** — cascada + el afinado de hueso fino/taxón (sección 2).

Las comparamos sobre la misma imagen de ejemplo:

In [ ]:
from shared.inference.predict_geometric import classify_image as geometric_classify

with open(sample, "rb") as fh:
    g = geometric_classify(fh)

print("Grupo:", g.get("prediction"))
print("Probabilidades:", g.get("probabilities"))
print("Punto en el PCA:", g.get("pca_point"))

In [ ]:
import pandas as pd

filas = []
for nombre, funcion, kwargs in [
    ("geométrico", geometric_classify, {}),
    ("cascada", cascade_classify, {"model_type": "cnn_only", "backend": "classic"}),
    ("2 etapas", classify_image_two_stage, {"backend": "classic"}),
]:
    with open(sample, "rb") as fh:
        try:
            salida = funcion(fh, **kwargs)
            filas.append({"ruta": nombre, "predicción": salida.get("prediction"),
                          "extra": salida.get("gate") or salida.get("stage2_prediction")})
        except Exception as exc:   # el modelo puede no estar entrenado en esta copia del repo
            filas.append({"ruta": nombre, "predicción": f"error: {type(exc).__name__}",
                          "extra": str(exc)[:60]})

pd.DataFrame(filas)

## 4. Cómo llega esto a una aplicación real: la API

Lo que acabamos de llamar directamente en Python como funciones es, también, exactamente lo que
expone la API en producción (`serving/backend/main.py`), que es a la que llama tanto la web como
una app de Streamlit. El flujo típico al subir una foto:

1. `POST /filter/clip` — ¿esto es un hueso, siquiera? (cuaderno 1, sección 6). Si no, fin.
2. `POST /classify` — mide con OpenCV, clasifica con el Random Forest, y proyecta el punto en el
   PCA 2D.
3. `POST /hybrid/classify` — la cascada de dos etapas sobre el modelo híbrido (grupo + hueso
   fino/taxón + posible `stage2_pending`).
4. `GET /pca/reference` — las ~1300 vistas de fondo que se dibujan en el scatter de la interfaz
   (se cachean en memoria con `lru_cache`, no se recalculan en cada petición).

Un detalle de diseño que vale la pena mencionar: cada módulo de inferencia carga sus modelos **una
sola vez, al importarse** — no en cada petición HTTP. Y si el modelo no está entrenado todavía en
ese entorno, falla con un mensaje explícito en vez de devolver un resultado silenciosamente vacío.

In [ ]:
# Llamadas de ejemplo contra la API ya levantada:
# uvicorn serving.backend.main:app --port 8000
ejemplos_curl = '''
curl http://localhost:8000/health
curl -X POST http://localhost:8000/classify        -F "file=@imagen.jpg"
curl -X POST http://localhost:8000/hybrid/classify -F "file=@imagen.jpg"
curl http://localhost:8000/pca/reference
'''
print(ejemplos_curl.strip())

## 5. Reproducibilidad: que todo esto se pueda rehacer, no solo que funcione hoy

Para que este proyecto no dependa de "en mi máquina funciona", cada pieza queda registrada en un
sistema pensado para eso:

- **DVC** (`dvc.yaml`): declara la etapa `train_geometric` con sus dependencias, parámetros y
  salidas exactas. `dvc repro` reconstruye el modelo automáticamente si cambió el dataset, el
  código o `params.yaml` — y no hace nada si no cambió nada, para no perder tiempo repitiendo
  trabajo ya hecho.
- **MLflow**: cada entrenamiento y cada evaluación quedan registrados con sus parámetros,
  métricas (de las dos evaluaciones del cuaderno 2) y tags, en experimentos separados por modelo
  para que no se mezclen entre sí.
- **Tests** (`pytest`): por ejemplo, `tests/test_group_splits.py` protege justo el problema que
  explicamos en el cuaderno 1 — que ningún espécimen se filtre entre train y test — y
  `tests/test_hybrid_endpoint.py` cubre el contrato del endpoint `/hybrid/classify`.
- **Historial** (`base_datos_osea/HISTORIAL.md`): un registro append-only de *por qué* se tomó
  cada decisión, no solo *qué* se cambió — el código y los commits dicen el qué, esto dice el
  porqué.

In [ ]:
run(["python", "-m", "pytest", "-q", "tests"], enabled=False)
run(["dvc", "repro"], enabled=False)
print("\nMLflow: mlflow ui --port 5000  (o http://localhost:5000 con el stack de Docker levantado)")

## Resumen de la serie

- **Cuaderno 1** — de mallas 3D y fotos reales curadas a tablas de features geométricas, con
  cuidado de no mezclar especímenes entre train y test.
- **Cuaderno 2** — dos modelos entrenados sobre esos datos (Random Forest y CNN híbrida),
  validados con semillas múltiples y evaluados contra tres bancos cada vez más exigentes.
- **Cuaderno 3** (este) — cómo esos modelos se combinan en una cascada de decisiones y se exponen
  como una API real.

El hilo que atraviesa los tres: en un problema con pocos datos reales disponibles, casi todas las
decisiones importantes (qué taxonomía usar, qué recetas de render mezclar, qué regla de corte
elegir, qué checkpoint quedarse) se validaron siempre contra **fotos reales**, nunca solo contra
el banco sintético limpio — porque ese es, al final, el único resultado que cuenta.